## Reminders projects

In [3]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [55]:
# Helper functions
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {"role": "user", "content": message.content if isinstance(message, Message) else message}
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {"role": "assistant", "content": message.content if isinstance(message, Message) else message}
    messages.append(assistant_message)


def chat(messages, system=None,tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools
    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [66]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [23]:
#get_current_datetime

from datetime import datetime
from anthropic.types import ToolParam # To avoid type errors

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
  if not date_format:
    raise ValueError("date_format cannot be empty")

  return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time from the server's system clock as a formatted string. Use this whenever the answer depends on the current date or time, such as today's date, the current time, or calculating how long until or since an event. Do not use it for historical dates or for converting between time zones. It returns the server's local time, with no time zone information. The format is controlled by the optional date_format parameter, which uses Python strftime codes. If date_format is omitted, the result looks like '2026-11-07 08:09:00'.",
  "strict": True,
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "minLength": 1,
        "default": "%Y-%m-%d %H:%M:%S",
        "description": "Optional Python strftime format string for the output, e.g. '%Y-%m-%d' for date only, '%H:%M' for time only, or '%A, %B %d, %Y' for a long date. Must be non-empty. Defaults to '%Y-%m-%d %H:%M:%S'."
      }
    },
    "required": [],
    "additionalProperties": False
  }
})

get_current_datetime()

'2026-11-07 08:26:28'

In [38]:
messages=[]
messages.append({"role":"user",
                 "content": "what is the exact time, formatted as HH:MM:SS?"})

response=client.messages.create(
    model=model,
    max_tokens=500,
    messages=messages,
    tools=[get_current_datetime_schema]
)
messages.append(
    {"role":"assistant",
     "content": response.content})

In [39]:
messages

[{'role': 'user', 'content': 'what is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_013NrQDzsVomVMXgYjswJcBF', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [40]:
result = get_current_datetime(**response.content[0].input)

In [41]:
messages.append(
    {"role":"user",
     "content":[
        {
            "type":"tool_result",
            "tool_use_id":response.content[0].id,
            "content":result,
            "is_error":False

        }
     ]
     }
)

In [42]:
messages

[{'role': 'user', 'content': 'what is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_013NrQDzsVomVMXgYjswJcBF', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_013NrQDzsVomVMXgYjswJcBF',
    'content': '08:39:57',
    'is_error': False}]}]

In [44]:
client.messages.create(
    model=model,
    max_tokens=500,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_011CgnqJtKGYGEDmUiP8Mzki', container=None, content=[TextBlock(citations=None, text='The exact time is **08:39:57**.', type='text')], diagnostics=None, model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=879, output_tokens=14, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'))

In [53]:
#Testing after upgrading add_user_message and add_assistant_message to accept Message objects
messages=[]
add_user_message(messages, "what is the exact time, formatted as HH:MM:SS?")
response=client.messages.create(
    model=model,
    max_tokens=500,
    messages=messages,
    tools=[get_current_datetime_schema]
)
add_assistant_message(messages,response)
messages

[{'role': 'user', 'content': 'what is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01UVV5JXTZSxNzTRF4dwa6Bw', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [63]:
#run tools
import json

def run_tool(tool_name,tool_input):
    if tool_name=="get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name=="add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name=="set_reminder":
        return set_reminder(**tool_input)
    
def run_tools(message):
    tool_requests = [
        block for block in message.content
        if block.type == "tool_use"
    ]
    tool_result_blocks=[]
    for tool_request in tool_requests:

        try:
            tool_output=run_tool(tool_request.name, tool_request.input)
            tool_result_block={
                "type":"tool_result",
                "tool_use_id":tool_request.id,
                "content":json.dumps(tool_output),
                "is_error":False
            }
        except Exception as e:
            tool_result_block={
                "type":"tool_result",
                "tool_use_id":tool_request.id,
                "content":f"Error: {e}",
                "is_error":True
            }
        tool_result_blocks.append(tool_result_block)
    return tool_result_blocks

In [64]:
#run conversation

def run_conversation(messages):
    while True:
        response=chat(messages,tools=[get_current_datetime_schema,add_duration_to_datetime_schema,set_reminder_schema,batch_tool_schema])
        add_assistant_message(messages,response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break
        tool_results=run_tools(response)
        add_user_message(messages,tool_results)
    return messages

In [68]:
messages=[]
add_user_message(messages, "Set a reminder for 3 days from now at 9 AM to take my medication.")
run_conversation(messages)

I'll help you set a reminder for 3 days from now at 9 AM. Let me first get the current date and time, then calculate 3 days from now.
Now I'll calculate the date 3 days from now at 9 AM and set the reminder.
Perfect! Now I'll set the reminder for that time.
----
Setting the following reminder for 2026-11-10T09:00:00:
Take my medication
----
Done! I've set a reminder for **Tuesday, November 10, 2026 at 9:00 AM** to take your medication. You'll receive a notification at that time.


[{'role': 'user',
  'content': 'Set a reminder for 3 days from now at 9 AM to take my medication.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for 3 days from now at 9 AM. Let me first get the current date and time, then calculate 3 days from now.", type='text'),
   ToolUseBlock(id='toolu_017xJCNtfhh7ibZ8zxvb6cnp', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_017xJCNtfhh7ibZ8zxvb6cnp',
    'content': '"2026-11-07 09:49:17"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll calculate the date 3 days from now at 9 AM and set the reminder.", type='text'),
   ToolUseBlock(id='toolu_019GGbeuFgrHj2vso2ep6Unj', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-11-07 09:00:00', 'duration': 3, 'unit': 'days', 'input_format

## ToolUse and Streaming

## Web Search tool

## Text Edit tool